# 04 — Post-Hoc Explainability: TempME Temporal Motif Discovery

### Academic Research Objectives:
1. **Explainable Dynamic Contagion:** Extract the most influential temporal interaction motifs explaining CryptoTGN predictions.
2. **Quantitative Validation Protocol:**
   - **Fidelity+:** Predictive degradation under top-$K\%$ explanatory edge ablation.
   - **Fidelity-:** Predictive retention under explanatory motif preservation.
   - **Sparsity & Temporal Entropy:** Characterization metrics of explanation quality.
3. **Econometric Cross-Validation:** Rank agreement (Spearman $\rho_{\text{rank}}$) between node explanation attribution and independent VAR-GFEVD NET spillovers and PageRank centralities.


In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

src_path = str(Path("../src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import torch
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import spearmanr

from cryptograph.data.universe import TARGET_UNIVERSE_30


## 1. Load Trained CryptoTGN Checkpoint and Target Prediction Events


In [2]:
import glob
import torch
from cryptograph.models.tgn import CryptoTGN

tgn_ckpts = sorted(glob.glob('../runs/*tgn*/best_tgn_checkpoint.pt'))
if tgn_ckpts:
    ckpt_file = tgn_ckpts[-1]
    print(f'Loading TGN checkpoint: {ckpt_file}')
    ckpt = torch.load(ckpt_file, map_location='cpu', weights_only=False)
    model = CryptoTGN(num_nodes=30, raw_msg_dim=9, node_feat_dim=16, memory_dim=64, embedding_dim=64, out_horizons=5)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    print('CryptoTGN model ready for explainability attribution.')
else:
    print('No checkpoint found.')


C:\Users\daksh\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading TGN checkpoint: ../runs\20261001-144534_tgn_hourly_contagion_s42_12e04d8\best_tgn_checkpoint.pt
CryptoTGN model ready for explainability attribution.


## 2. Extract Temporal Motifs via TempME Explainer


In [3]:
exp_csv = Path('../reports/tempme_explanations.csv')
if exp_csv.exists():
    df_exp = pd.read_csv(exp_csv)
    print(f'Loaded {len(df_exp)} TempME test explanation instances.')
    print(df_exp[['timestep', 'target_asset', 'fidelity_plus', 'fidelity_minus', 'sparsity', 'top_explanatory_edges']].head(10).to_string())
else:
    print('reports/tempme_explanations.csv not found.')


Loaded 8 TempME test explanation instances.
   timestep target_asset  fidelity_plus  fidelity_minus  sparsity                              top_explanatory_edges
0     20133         AAVE       5.265797        0.835491  0.993333      LTC->AAVE(0.68);BTC->ETC(0.33);OP->ATOM(0.31)
1     20134         AAVE       4.544519        2.712677  0.993333      LTC->AAVE(0.71);XLM->DOT(0.33);ICP->UNI(0.33)
2     20135         AAVE       0.000000        0.000000  1.000000  SHIB->ADA(0.34);UNI->ATOM(0.33);INJ->RENDER(0.33)
3     20136         AAVE       0.000000        0.000000  1.000000  AVAX->PEPE(0.33);XRP->ADA(0.32);RENDER->FIL(0.32)
4     20137         AAVE       0.000000        0.000000  1.000000       INJ->DOT(0.33);ETC->UNI(0.33);FET->ADA(0.32)
5     20138         AAVE       0.000000        0.000000  1.000000      XRP->ETH(0.33);BNB->AVAX(0.32);SOL->ETH(0.32)
6     20139         AAVE       0.000000        0.000000  1.000000    XLM->DOT(0.33);LINK->DOGE(0.33);DOGE->DOT(0.32)
7     20140         

## 3. Visualize Attributed Shock Transmission Subgraphs


In [4]:
import networkx as nx

if exp_csv.exists() and len(df_exp) > 0:
    sample_row = df_exp.iloc[0]
    target = sample_row['target_asset']
    edge_str = sample_row['top_explanatory_edges']
    print(f'Explaining contagion prediction for {target}:')
    print(f'Top motif edges: {edge_str}')
    
    G_exp = nx.DiGraph()
    for item in edge_str.split(';'):
        if '->' in item:
            uv, w_str = item.split('(')
            u, v = uv.split('->')
            w = float(w_str.rstrip(')'))
            G_exp.add_edge(u, v, weight=w)
            
    pos = nx.spring_layout(G_exp, seed=42)
    edge_x = []
    edge_y = []
    for u, v in G_exp.edges():
        x0, y0 = pos[u]
        x1, y1 = pos[v]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])
        
    node_x = [pos[n][0] for n in G_exp.nodes()]
    node_y = [pos[n][1] for n in G_exp.nodes()]
    
    fig = go.Figure(
        data=[
            go.Scatter(x=edge_x, y=edge_y, line=dict(width=2, color='#888'), hoverinfo='none', mode='lines'),
            go.Scatter(x=node_x, y=node_y, mode='markers+text', text=list(G_exp.nodes()), textposition='top center', marker=dict(size=14, color='#1f77b4'))
        ],
        layout=go.Layout(
            title=f'TempME Attributed Subgraph for Shock Target: {target}',
            showlegend=False,
            template='plotly_white',
            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False)
        )
    )
    fig.show()


Explaining contagion prediction for AAVE:
Top motif edges: LTC->AAVE(0.68);BTC->ETC(0.33);OP->ATOM(0.31)


## 4. Quantitative Explanation Validation: Fidelity+, Fidelity-, and Spearman Rank Agreement


In [5]:
val_csv = Path('../reports/explanation_validation.csv')
if val_csv.exists():
    df_val = pd.read_csv(val_csv)
    print('=== TempME Explanation Validation Summary ===')
    print(df_val.T.to_string())
    print('\nKey Findings:')
    print(f'1. Fidelity+ (Ablation Degradation): {df_val["mean_fidelity_plus"].values[0]:.4f}')
    print(f'2. Fidelity- (Attribution Retention): {df_val["mean_fidelity_minus"].values[0]:.4f}')
    print(f'3. Sparsity (% edges pruned): {df_val["mean_sparsity"].values[0]*100:.2f}%')
    print(f'4. Spearman Rank Agreement with DY NET Spillover: {df_val["mean_spearman_dy_net"].values[0]:.4f}')
else:
    print('reports/explanation_validation.csv not found.')


=== TempME Explanation Validation Summary ===
                                0
mean_fidelity_plus       1.226290
mean_fidelity_minus      0.443521
mean_sparsity            0.998333
mean_motif_entropy       5.006238
mean_spearman_dy_net     0.441824
mean_spearman_pagerank  -0.252781
sample_events_evaluated  8.000000

Key Findings:
1. Fidelity+ (Ablation Degradation): 1.2263
2. Fidelity- (Attribution Retention): 0.4435
3. Sparsity (% edges pruned): 99.83%
4. Spearman Rank Agreement with DY NET Spillover: 0.4418
